In [1]:
# Impoort thư viện
import os
import shutil
from pathlib import Path
from collections import Counter
import concurrent.futures
from tqdm.auto import tqdm

In [2]:
DATASET1_DIR = "Dataset 1_organized"
DATASET2_DIR = "Dataset 2_clean"
OUTPUT_DIR = "Merged_Dataset"

# Chỉ gộp tập train để làm dữ liệu huấn luyện, giữ nguyên tập test của 2 bộ.
SPLITS = ["train"] 
# Chọn các định dạng ảnh hợp lệ
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Class Map: gom nhiều class nhỏ thành 8 class lớn
CLASS_MAP = {
    # DATASET 1 -> Gom về các nhóm lớn
    "aerosol_cans": "Metal",
    "aluminum_food_cans": "Metal",
    "aluminum_soda_cans": "Metal",
    "steel_food_cans": "Metal",

    "plastic_cup_lids": "Plastic",
    "plastic_detergent_bottles": "Plastic",
    "plastic_food_containers": "Plastic",
    "plastic_shopping_bags": "Plastic",
    "plastic_soda_bottles": "Plastic",
    "plastic_straws": "Plastic",
    "plastic_trash_bags": "Plastic",
    "plastic_water_bottles": "Plastic",
    "disposable_plastic_cutlery": "Plastic",
    "styrofoam_cups": "Plastic",
    "styrofoam_food_containers": "Plastic",

    "cardboard_boxes": "Paper",
    "cardboard_packaging": "Paper",
    "magazines": "Paper",
    "newspaper": "Paper",
    "office_paper": "Paper",
    "paper_cups": "Paper",

    "glass_beverage_bottles": "Glass",
    "glass_cosmetic_containers": "Glass",
    "glass_food_jars": "Glass",

    "coffee_grounds": "Organic",
    "eggshells": "Organic",
    "food_waste": "Organic",
    "tea_bags": "Organic",

    "clothing": "Textiles",
    "shoes": "Textiles",

    # DATASET 2 -> Đồng nhất về cùng 8 lớp tiêu chuẩn
    "E-waste": "Electronics",
    "battery waste": "Electronics",
    "light bulbs": "Electronics",

    "automobile wastes": "Other",

    "glass waste": "Glass",
    "metal waste": "Metal",
    "organic waste": "Organic",
    "paper waste": "Paper",
    "plastic waste": "Plastic",
}

**Phân loại hình ảnh rác thải thành 8 lớp: Metal, Plastic, Paper, Organic, Electronics, Glass, Textiles, Others***

***CÁC LOẠI RÁC CÓ TRONG DATASET***

**1. Metal**

*Định nghĩa:* Vật thể được làm chủ yếu từ kim loại hoặc hợp kim, bao gồm nhôm, thép, sắt, đồng và thiếc. Đây là nhóm vật liệu có thể tái chế vô hạn lần mà không mất đi tính chất vật lý.

*Tiêu chí gán nhãn:* Kim loại chiếm >50% khối lượng vật thể, không chứa linh kiện điện tử bên trong

*Các loại rác thuộc nhóm này:* aerosol_cans, aluminum_food_cans, aluminum_soda_cans, steel_food_cans

**2. Plastic**

*Định nghĩa:* Vật thể được làm chủ yếu từ polymer tổng hợp, bao gồm các nhựa phổ biến như PET, HDPE, PVC, LDPE, PP, PS và EPS (xốp).

*Tiêu chí gán nhãn:* Polymer tổng hợp chiếm >50% khối lượng, không có nguồn gốc sinh học,không chứa linh kiện điện tử

*Các loại rác thuộc nhóm này:* plastic_cup_lids, plastic_detergent_bottles, plastic_food_containers, plastic_shopping_bags, plastic_soda_bottles, plastic_straws, plastic_trash_bags, plastic_water_bottles, disposable_plastic_cutlery,styrofoam_cups, styrofoam_food_containers

**3. Paper**

*Định nghĩa:* Vật thể được làm chủ yếu từ thực vật, bao gồm giấy in, bìa carton, báo, tạp chí và bao bì giấy. Với các vật liệu hỗn hợp, nhưng thành phần giấy phải chiếm phần nhiều

*Tiêu chí gán nhãn:* Nguồn gốc từ gỗ hoặc thực vật hoặc thành phần thực vật đã qua xử lý chiếm trên 50% khối lượng, không phải rác thực phẩm hay chất hữu cơ phân hủy

*Các loại rác thuộc nhóm này:* cardboard_boxes, cardboard_packaging, magazines, newspaper, office_paper, paper_cups

**4. Organic**

*Định nghĩa:* Vật thể có nguồn gốc sinh học, có thể phân hủy tự nhiên bởi vi sinh vật trong điều kiện phù hợp. Bao gồm thức ăn thừa, phế phẩm thực phẩm, và các chất hữu cơ từ thực vật hoặc động vật.

*Tiêu chí gán nhãn:* Có khả năng phân hủy sinh học, nguồn gốc từ sinh vật sống (thực vật, động vật) và không qua xử lý hóa học biến tính vật liệu

*Các loại rác thuộc nhóm này:* coffee_grounds, eggshells, food_waste, tea_bags

**5. Glass**

*Định nghĩa:* Vật thể được làm chủ yếu từ thủy tinh — vật liệu vô cơ, phi kim loại, được tạo thành bằng cách nấu chảy cát silica ở nhiệt độ cao. Thủy tinh có thể tái chế vô hạn lần mà không mất chất lượng.

*Tiêu chí gán nhãn:* Thủy tinh chiếm >50% khối lượng vật thể, vật liệu vô cơ, trong suốt hoặc màu

*Các loại rác thuộc nhóm này:* glass_beverage_bottles, glass_cosmetic_containers, glass_food_jars

**6. Textiles**

*Định nghĩa:* Vật thể được làm từ sợi dệt, bao gồm vải tự nhiên (cotton, len, lụa), vải tổng hợp (polyester, nylon), da thuộc, và các sản phẩm may mặc

*Tiêu chí gán nhãn:* Sợi dệt hoặc da chiếm >50% bề mặt/khối lượng, sản phẩm may mặc, giày dép, phụ kiện vải, không phải vật liệu giấy hay nhựa thuần túy

*Các loại rác thuộc nhóm này:* clothing, shoes

**7. Electronics**

*Định nghĩa:* Thiết bị hoặc linh kiện có chứa mạch điện tử, pin, hoặc chất nguy hại cần quy trình xử lý đặc biệt. Nhóm này không xác định theo vật liệu bên ngoài, mà theo phương thức cuối vòng đời nó là cái gì.

*Tiêu chí gán nhãn:* Có chứa linh kiện điện tử hoặc pin, hoặc chứa chất nguy hại (thủy ngân, chì, cadmium), yêu cầu xử lý theo quy định WEEE/e-waste

*Các loại rác thuộc nhóm này:* E-waste, battery waste, light bulbs

**8. Others**

*Định nghĩa:* Vật thể không thể phân loại rõ ràng vào 7 lớp trên do: thành phần vật liệu hỗn hợp phức tạp không có thành phần nào chiếm ưu thế hoặc do không thể phân loại vào các lớp bên trên

*Tiêu chí gán nhãn:* Không có vật liệu đơn nào chiếm >50% khối lượng, hoặc vật liệu hỗn hợp không thể tách rời trong thực tế, hoặc không fit tiêu chí của 7 lớp trên sau khi áp dụng đầy đủ quy tắc ưu tiên, hoặc nếu không phân loại được do hỗn loạn 

*Các loại rác thuộc nhóm này:* automobile wastes

In [3]:
# Hàm copy file ảnh
def copy_single_file(task):
    src_path, dst_path = task

    try:
        shutil.copy2(src_path, dst_path)
        return True

    except Exception as e:
        print(f"Lỗi copy {src_path.name}: {e}")
        return False

# Chuẩn bị danh sách các file cần copy 
def prepare_merge_tasks(
    dataset_dir: Path,
    dataset_prefix: str,
    output_dir: Path
):

    tasks = []

    for split in SPLITS:
        split_dir = dataset_dir / split

        if not split_dir.exists():
            continue
        
        # Lấy danh sách các class
        classes = sorted([
            d.name for d in split_dir.iterdir()
            if d.is_dir()
        ])

        for cls in classes:
            # Skip class chưa mapping
            if cls not in CLASS_MAP:
                continue

            # Đổi tên class nhỏ thành class lớn
            super_cls = CLASS_MAP[cls]

            #Tạo đường dẫn nguồn và đích
            src_cls_dir = split_dir / cls
            dst_cls_dir = output_dir / split / super_cls

            # Tạo thư mục đích
            dst_cls_dir.mkdir(parents=True, exist_ok=True)

            # Lấy file ảnh hợp lệ
            img_files = [
                f for f in src_cls_dir.iterdir()
                if (
                    f.is_file()
                    and f.suffix.lower() in VALID_EXTENSIONS
                )
            ]
            # Duyệt từng ảnh:
            for img_path in img_files:

                # Đổi tên file
                dst_name = (f"{dataset_prefix}_"f"{img_path.name}")
                # Tạo đường dẫn đích
                dst_path = dst_cls_dir / dst_name

                tasks.append((img_path, dst_path))

    return tasks

In [4]:
def main():
    dataset1_dir = Path(DATASET1_DIR)
    dataset2_dir = Path(DATASET2_DIR)
    output_dir = Path(OUTPUT_DIR)

    # Xóa output cũ 
    if output_dir.exists():
        shutil.rmtree(output_dir)
   
    # Chuẩn bị danh sách file cần gộp
    tasks_ds1 = prepare_merge_tasks(dataset1_dir, "ds1", output_dir)
    tasks_ds2 = prepare_merge_tasks(dataset2_dir, "ds2", output_dir)
    
    all_tasks = tasks_ds1 + tasks_ds2

    # Đếm tổng số ảnh
    total_images = len(all_tasks)
    print(f"Tổng số ảnh: {total_images}")

    # Kiểm tra dataset rỗng
    if total_images == 0:
        print("Không tìm thấy ảnh.")
        return

    # Gộp dữ liệu
    with concurrent.futures.ThreadPoolExecutor(
        max_workers=min(32, os.cpu_count() * 2)
    ) as executor:

        results = list(tqdm(
            executor.map(copy_single_file, all_tasks),
            total=total_images,
            desc="Gộp ảnh"
        ))

    success_count = sum(results)
    print(f"Gộp thành công: {success_count}/{total_images}")

    # Thống kê số lượng ảnh copy thành công
    stats = Counter()

    for is_success, (_, dst_path) in zip(results, all_tasks):

        if is_success:
            cls_name = dst_path.parent.name
            stats[cls_name] += 1

    print("\nTHỐNG KÊ DATASET")

    for cls in sorted(stats):
        print(f"{cls:<12}: {stats[cls]}")

    print(f"\nTổng ảnh train: {success_count}")


if __name__ == "__main__":
    main()

Tổng số ảnh: 20700


Gộp ảnh:   0%|          | 0/20700 [00:00<?, ?it/s]

Gộp thành công: 20700/20700

THỐNG KÊ DATASET
Electronics : 2508
Glass       : 2218
Metal       : 2824
Organic     : 2042
Other       : 866
Paper       : 3750
Plastic     : 5692
Textiles    : 800

Tổng ảnh train: 20700
